# 07 · n-step Bootstrapping & Planning
**TABULAR METHODS** · ⏱ 50 min · 🧪 3 exercises · ⭐ 440 XP

*The dial between TD and Monte Carlo (n-step returns, eligibility traces), then learned models and Dyna-Q — imagining experience to learn faster.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "07-nstep-and-planning"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

The last two lessons gave you two extremes. Monte Carlo waits for the whole return; TD(0)
looks one reward ahead and then trusts its own guess. Neither is best in general. This lesson
does two things with that observation:

1. It builds a **dial** between them — n-step returns and eligibility traces — so you can pick
   how far to look before bootstrapping.
2. It asks a different question: if every real step is precious, can the agent **reuse** it?
   Learning a model of the world and planning with it — **Dyna** — answers yes.

## 01 · n-step returns

Write out the targets of the methods you know, for the state visited at time t:

```text
TD(0):     Gₜ:ₜ₊₁ = Rₜ₊₁ + γ V(Sₜ₊₁)
2-step:    Gₜ:ₜ₊₂ = Rₜ₊₁ + γ Rₜ₊₂ + γ² V(Sₜ₊₂)
n-step:    Gₜ:ₜ₊ₙ = Rₜ₊₁ + γ Rₜ₊₂ + … + γⁿ⁻¹ Rₜ₊ₙ + γⁿ V(Sₜ₊ₙ)
MC:        Gₜ     = Rₜ₊₁ + γ Rₜ₊₂ + … + γᵀ⁻ᵗ⁻¹ R_T
```

The **n-step return** uses n real rewards, then bootstraps from the estimate n steps later.
If the episode ends before that (t + n ≥ T), there is nothing to bootstrap from and the n-step
return is just the ordinary return. n = 1 is TD(0); n = ∞ is Monte Carlo.

```text
V(Sₜ) ← V(Sₜ) + α [ Gₜ:ₜ₊ₙ − V(Sₜ) ]
```

There is one practical wrinkle: Gₜ:ₜ₊ₙ needs Rₜ₊ₙ, which only arrives at time t + n. So the
update for time τ is made at time τ + n − 1, and after the episode ends you must flush the last
n − 1 pending updates. Store the states and rewards of the episode and index carefully.

> **ℹ️ NOTE** · **The bias–variance dial**
>
> Each real reward in the target adds a little variance (it is a random sample); each bootstrap
> adds a little bias (V is only a guess). Small n: low variance, high bias. Large n: the reverse.
> On most problems the error is lowest somewhere in the middle — and the best α shrinks as n grows,
> because the target gets noisier.

### Worked example

The deterministic chain from lesson 6 (0 → 1 → 2 → terminal, reward 1 at the end, γ = 1,
α = 0.5, V = 0). With **n = 2**, in the very first episode:

| τ | update made at time | target | new V(S_τ) |
|---|---|---|---|
| 0 | 1 | R₁ + R₂ + V(S₂) = 0 + 0 + 0 | 0 |
| 1 | 2 | R₂ + R₃ = 0 + 1 (episode ended) | 0.5 |
| 2 | after the end | R₃ = 1 | 0.5 |

The reward reaches two states back in one episode — TD(0) managed one, MC all three.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("07-nstep-and-planning#0", {"q": "Rewards R₁ = 1, R₂ = 0, R₃ = 2, γ = 0.5, V(S₃) = 4, episode not over. What is the 3-step return G₀:₃?", "options": ["3.0", "2.0", "1.5", "7.0"], "answer": 1, "explain": "1 + 0.5·0 + 0.25·2 + 0.125·4 = 1 + 0 + 0.5 + 0.5 = 2.0"})

---
### 🧪 Exercise 1 · n-step TD prediction · ⭐ **150 XP**

Generalise TD(0) to look *n* rewards ahead before bootstrapping.

1. `nstep_target(rewards, gamma, bootstrap)` → `Σᵢ γⁱ·rewards[i] + γᵏ·bootstrap` with `k = len(rewards)`.
2. `nstep_td(env, policy, n_episodes, n, alpha, gamma=1.0, init=0.0, V=None)` → `V` (numpy array; update the given one in place if passed). Implement the algorithm of Sutton & Barto §7.1: store `states` and `rewards` as the episode unfolds (`rewards[0]` is a dummy so `rewards[i]` is Rᵢ). At time *t* the state from time **τ = t − n + 1** gets updated:
   - `G = nstep_target(rewards[τ+1 : min(τ+n, T)+1], γ, V[states[τ+n]] if τ+n < T else 0)`
   - `V[states[τ]] += α·(G − V[states[τ]])`

   Keep going after the episode ends (T = time of the last step) until τ = T − 1, so the last n − 1 states also get their update. Treat `terminated or truncated` as the end.

The check works through a 3-step chain by hand for several n (n = 1 must equal TD(0); n ≥ 3 must equal Monte Carlo), then reproduces the key result of Figure 7.2 on the 19-state random walk. **Run** draws the full figure.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 07-nstep-and-planning/01-nstep-td
import random

import numpy as np
import rlhud


def nstep_target(rewards, gamma, bootstrap):
    """G = r1 + gamma*r2 + ... + gamma^(k-1)*rk + gamma^k * bootstrap,  where k = len(rewards).
    Pass bootstrap=0.0 when the episode ended inside the window."""
    # TODO
    raise NotImplementedError


def nstep_td(env, policy, n_episodes, n, alpha, gamma=1.0, init=0.0, V=None):
    """n-step TD prediction of v_pi (Sutton & Barto, section 7.1).

    Returns V (numpy array, length env.n_states). If V is given, it is updated in place.
    The update for the state visited at time tau happens once R_{tau+n} (or the episode end)
    is known:  V(S_tau) += alpha * (G_{tau:tau+n} - V(S_tau)).
    An episode ends on terminated or truncated (both treated as the end: no bootstrap past it).
    """
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        s, _ = env.reset()
        states, rewards = [s], [0.0]   # rewards[i] holds R_i, so rewards[0] is a dummy
        T = float("inf")
        t = 0
        # TODO: while True:
        #   - if t < T: step with policy(states[t]); append to states/rewards; on the end set T = t + 1
        #   - tau = t - n + 1; if tau >= 0: build the n-step target G and update V[states[tau]]
        #   - stop once tau == T - 1, else t += 1
        raise NotImplementedError
    return V


# ---------------------------------------------------------------- provided
class RandomWalk:
    """States 1..n are non-terminal, 0 and n+1 terminal; start in the middle.
    Action 0 = left, 1 = right. Exit right -> right_reward, exit left -> left_reward."""

    def __init__(self, n=19, left_reward=-1.0, right_reward=1.0):
        self.n = n
        self.n_states = n + 2
        self.n_actions = 2
        self.left_reward, self.right_reward = left_reward, right_reward

    def reset(self, seed=None):
        self.s = (self.n + 1) // 2
        return self.s, {}

    def step(self, a):
        self.s += 1 if a == 1 else -1
        if self.s == 0:
            return 0, self.left_reward, True, False, {}
        if self.s == self.n + 1:
            return self.s, self.right_reward, True, False, {}
        return self.s, 0.0, False, False, {}

    def true_values(self):
        k = np.arange(1, self.n + 1)
        return self.left_reward + (self.right_reward - self.left_reward) * k / (self.n + 1)


def coin_flip(s):
    return random.randrange(2)


def avg_rms(n, alpha, runs=50, episodes=10, seed=0):
    """Mean RMS error over the first `episodes` episodes and `runs` runs (the S&B Figure 7.2 metric)."""
    env = RandomWalk()
    truth = env.true_values()
    random.seed(seed)
    total = 0.0
    for _ in range(runs):
        V = np.zeros(env.n_states)
        for _ in range(episodes):
            nstep_td(env, coin_flip, 1, n, alpha, V=V)
            total += np.sqrt(np.mean((V[1:-1] - truth) ** 2))
    return total / (runs * episodes)


if __name__ == "__main__":
    alphas = [0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0]
    ns = [1, 2, 4, 8, 16, 32]
    rows = []
    for i, n in enumerate(ns):
        errs = [avg_rms(n, a) for a in alphas]
        rlhud.plot(errs, x=alphas, label=f"n = {n}", title="19-state random walk: avg RMS over first 10 episodes",
                   xlabel="α", ylabel="RMS error")
        best = int(np.argmin(errs))
        rows.append([n, alphas[best], round(errs[best], 3)])
        rlhud.progress(i, len(ns), "n values")
    rlhud.table(rows, headers=["n", "best α", "RMS at best α"], title="Best step size per n")
    print("Intermediate n (≈ 4–8) beats both one-step TD (n = 1) and near-Monte-Carlo (n = 32).")

In [ ]:
#@title ✅ **Check** · n-step TD prediction (+150 XP) — tests the code you last ran above
rlcolab.check("07-nstep-and-planning/01-nstep-td")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Initialise `T = float('inf')` and `t = 0`. Each iteration: if `t < T`, take a step and append to `states`/`rewards`, setting `T = t + 1` when the episode ends. Then compute τ and update if τ ≥ 0. Break when `τ == T − 1`; otherwise `t += 1`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Indices are the whole difficulty. The rewards that follow time τ are R_τ+1 … R_min(τ+n, T), i.e. the slice `rewards[tau + 1 : min(tau + n, T) + 1]`. Bootstrap from `V[states[tau + n]]` only if `tau + n < T` — otherwise the window already reaches the end of the episode. (Once T is set it is an int, so `min` returns an int you can slice with.)


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

if tau >= 0:
    end = min(tau + n, T)
    boot = V[states[tau + n]] if tau + n < T else 0.0
    G = nstep_target(rewards[tau + 1:end + 1], gamma, boot)
    V[states[tau]] += alpha * (G - V[states[tau]])
if tau == T - 1: break
t += 1

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("07-nstep-and-planning/01-nstep-td")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 07-nstep-and-planning/01-nstep-td
# Reference solution
import random

import numpy as np
import rlhud


def nstep_target(rewards, gamma, bootstrap):
    """G = r1 + gamma*r2 + ... + gamma^(k-1)*rk + gamma^k * bootstrap,  where k = len(rewards).
    Pass bootstrap=0.0 when the episode ended inside the window."""
    G = 0.0
    for i, r in enumerate(rewards):
        G += (gamma ** i) * r
    return G + (gamma ** len(rewards)) * bootstrap


def nstep_td(env, policy, n_episodes, n, alpha, gamma=1.0, init=0.0, V=None):
    """n-step TD prediction of v_pi (Sutton & Barto, section 7.1).

    Returns V (numpy array, length env.n_states). If V is given, it is updated in place.
    The update for the state visited at time tau happens once R_{tau+n} (or the episode end)
    is known:  V(S_tau) += alpha * (G_{tau:tau+n} - V(S_tau)).
    An episode ends on terminated or truncated (both treated as the end: no bootstrap past it).
    """
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        s, _ = env.reset()
        states, rewards = [s], [0.0]   # rewards[i] holds R_i, so rewards[0] is a dummy
        T = float("inf")
        t = 0
        while True:
            if t < T:
                s, r, terminated, truncated, _ = env.step(policy(states[t]))
                states.append(s)
                rewards.append(r)
                if terminated or truncated:
                    T = t + 1
            tau = t - n + 1               # the time whose estimate is updated now
            if tau >= 0:
                end = min(tau + n, T)
                boot = V[states[tau + n]] if tau + n < T else 0.0
                G = nstep_target(rewards[tau + 1:end + 1], gamma, boot)
                V[states[tau]] += alpha * (G - V[states[tau]])
            if tau == T - 1:
                break
            t += 1
    return V


# ---------------------------------------------------------------- provided
class RandomWalk:
    """States 1..n are non-terminal, 0 and n+1 terminal; start in the middle.
    Action 0 = left, 1 = right. Exit right -> right_reward, exit left -> left_reward."""

    def __init__(self, n=19, left_reward=-1.0, right_reward=1.0):
        self.n = n
        self.n_states = n + 2
        self.n_actions = 2
        self.left_reward, self.right_reward = left_reward, right_reward

    def reset(self, seed=None):
        self.s = (self.n + 1) // 2
        return self.s, {}

    def step(self, a):
        self.s += 1 if a == 1 else -1
        if self.s == 0:
            return 0, self.left_reward, True, False, {}
        if self.s == self.n + 1:
            return self.s, self.right_reward, True, False, {}
        return self.s, 0.0, False, False, {}

    def true_values(self):
        k = np.arange(1, self.n + 1)
        return self.left_reward + (self.right_reward - self.left_reward) * k / (self.n + 1)


def coin_flip(s):
    return random.randrange(2)


def avg_rms(n, alpha, runs=50, episodes=10, seed=0):
    """Mean RMS error over the first `episodes` episodes and `runs` runs (the S&B Figure 7.2 metric)."""
    env = RandomWalk()
    truth = env.true_values()
    random.seed(seed)
    total = 0.0
    for _ in range(runs):
        V = np.zeros(env.n_states)
        for _ in range(episodes):
            nstep_td(env, coin_flip, 1, n, alpha, V=V)
            total += np.sqrt(np.mean((V[1:-1] - truth) ** 2))
    return total / (runs * episodes)


if __name__ == "__main__":
    alphas = [0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0]
    ns = [1, 2, 4, 8, 16, 32]
    rows = []
    for i, n in enumerate(ns):
        errs = [avg_rms(n, a) for a in alphas]
        rlhud.plot(errs, x=alphas, label=f"n = {n}", title="19-state random walk: avg RMS over first 10 episodes",
                   xlabel="α", ylabel="RMS error")
        best = int(np.argmin(errs))
        rows.append([n, alphas[best], round(errs[best], 3)])
        rlhud.progress(i, len(ns), "n values")
    rlhud.table(rows, headers=["n", "best α", "RMS at best α"], title="Best step size per n")
    print("Intermediate n (≈ 4–8) beats both one-step TD (n = 1) and near-Monte-Carlo (n = 32).")

### n-step SARSA

Control works the same way with action values. Bootstrap from Q(Sₜ₊ₙ, Aₜ₊ₙ):

```text
Gₜ:ₜ₊ₙ = Rₜ₊₁ + … + γⁿ⁻¹ Rₜ₊ₙ + γⁿ Q(Sₜ₊ₙ, Aₜ₊ₙ)        Q(Sₜ, Aₜ) ← Q(Sₜ, Aₜ) + α [ Gₜ:ₜ₊ₙ − Q(Sₜ, Aₜ) ]
```

Picture an agent that wanders a grid and finally finds the goal. One-step SARSA strengthens only
the very last action of that path. 10-step SARSA strengthens the last ten. The longer the reward
is delayed, the bigger the win. (n-step *off-policy* methods need importance-sampling ratios over
the n actions again, or the tree-backup algorithm that avoids them.)

## 02 · Eligibility traces and TD(λ)

Choosing one n is arbitrary. Why not average **all** of them? The **λ-return** weights the
n-step return by (1 − λ)λⁿ⁻¹ — weights that sum to 1 and fade geometrically:

```text
Gₜ^λ = (1 − λ) Σₙ λⁿ⁻¹ Gₜ:ₜ₊ₙ
```

λ = 0 gives TD(0); λ = 1 gives Monte Carlo. That is the **forward view**: elegant, but it needs
the future, so it cannot be computed online.

The **backward view** gets (almost) the same updates online. Each state keeps an
**eligibility trace** e(s): a short-term memory of how recently and how often it was visited.

```text
δₜ = Rₜ₊₁ + γ V(Sₜ₊₁) − V(Sₜ)
e(Sₜ) ← e(Sₜ) + 1
V(s) ← V(s) + α δₜ e(s)        for every s
e(s) ← γ λ e(s)                for every s
```

When something surprising happens (a large δ), *all* recently visited states share the credit,
in proportion to their traces. Think of a rat that gets a shock after a bell rang twice and a
light flashed once: the bell is more "eligible" for blame. This is the trick that later lets
actor-critic and GAE assign credit across many steps.

> **🔑 KEY IDEA** · **One dial, two mechanisms**
>
> n-step returns and TD(λ) both interpolate between TD (small n, λ = 0) and Monte Carlo
> (n = ∞, λ = 1). n-step does it with a fixed window; traces do it with an exponentially
> fading memory, updated online with one TD error per step.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("07-nstep-and-planning#1", {"q": "With accumulating traces, γ = 1 and λ = 0.5, a state was visited 1 and 3 steps ago (nothing else). What is its trace now, just before the current update?", "options": ["1.5", "0.625", "0.75", "0.5"], "answer": 1, "explain": "Each visit adds 1, then decays by γλ = 0.5 per step: 0.5¹ + 0.5³ = 0.5 + 0.125 = 0.625."})

---
### 🧪 Exercise 2 · TD(λ) with eligibility traces · ⭐ **120 XP**

Implement the backward view of TD(λ): one TD error per step, credited to *every* recently visited state through a decaying **eligibility trace**.

`td_lambda(env, policy, n_episodes, lam, alpha, gamma=1.0, init=0.0, V=None)` returns `V` (update the given array in place if passed). At the start of each episode set `e = np.zeros(env.n_states)`. After every step:

1. `δ = r + γ·V[s2] − V[s]` (no `γ·V[s2]` term if `terminated`)
2. `e[s] += 1` — accumulating trace
3. `V += α·δ·e` — a vector update of **all** states
4. `e *= γ·λ`

The check walks through the 3-step chain for λ = 0 (must equal TD(0)), λ = 0.5 and λ = 1 (equals MC here), then compares λ = 0.8 against λ = 0 on the 19-state random walk. **Run** sweeps α for several λ (cf. S&B Figure 12.6) and draws a trace decaying over time. Notice what happens at λ = 1 with a large α.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 07-nstep-and-planning/02-td-lambda
import random

import numpy as np
import rlhud


def td_lambda(env, policy, n_episodes, lam, alpha, gamma=1.0, init=0.0, V=None):
    """Online TD(lambda) prediction with ACCUMULATING eligibility traces.

    Returns V (numpy array, length env.n_states); if V is given it is updated in place.
    Every step:
        delta = R + gamma * V(S') - V(S)        (V(S') counts as 0 if terminated)
        e(S) += 1                               (bump the trace of the state just left)
        V    += alpha * delta * e               (EVERY state, in proportion to its trace)
        e    *= gamma * lam                     (all traces fade)
    Traces start at zero at the beginning of every episode.
    """
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        e = np.zeros(env.n_states)
        s, _ = env.reset()
        # TODO: loop over steps: compute delta, bump e[s], update ALL of V, decay e;
        #       stop on terminated or truncated
        raise NotImplementedError
    return V


# ---------------------------------------------------------------- provided
class RandomWalk:
    """States 1..n are non-terminal, 0 and n+1 terminal; start in the middle.
    Action 0 = left, 1 = right. Exit right -> right_reward, exit left -> left_reward."""

    def __init__(self, n=19, left_reward=-1.0, right_reward=1.0):
        self.n = n
        self.n_states = n + 2
        self.n_actions = 2
        self.left_reward, self.right_reward = left_reward, right_reward

    def reset(self, seed=None):
        self.s = (self.n + 1) // 2
        return self.s, {}

    def step(self, a):
        self.s += 1 if a == 1 else -1
        if self.s == 0:
            return 0, self.left_reward, True, False, {}
        if self.s == self.n + 1:
            return self.s, self.right_reward, True, False, {}
        return self.s, 0.0, False, False, {}

    def true_values(self):
        k = np.arange(1, self.n + 1)
        return self.left_reward + (self.right_reward - self.left_reward) * k / (self.n + 1)


def coin_flip(s):
    return random.randrange(2)


def avg_rms(lam, alpha, runs=50, episodes=10, seed=0):
    """Mean RMS error over the first `episodes` episodes and `runs` runs on the 19-state walk."""
    env = RandomWalk()
    truth = env.true_values()
    random.seed(seed)
    total = 0.0
    for _ in range(runs):
        V = np.zeros(env.n_states)
        for _ in range(episodes):
            td_lambda(env, coin_flip, 1, lam, alpha, V=V)
            total += np.sqrt(np.mean((V[1:-1] - truth) ** 2))
    return total / (runs * episodes)


if __name__ == "__main__":
    alphas = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
    lams = [0.0, 0.4, 0.8, 0.9, 0.95, 1.0]
    rows = []
    for i, lam in enumerate(lams):
        with np.errstate(all="ignore"):                       # large α with λ ≈ 1 can diverge
            errs = [avg_rms(lam, a) for a in alphas]
        errs = [e if np.isfinite(e) and e < 0.6 else 0.6 for e in errs]   # clip for a readable plot
        rlhud.plot(errs, x=alphas, label=f"λ = {lam}", title="TD(λ) on the 19-state walk: avg RMS, first 10 episodes",
                   xlabel="α", ylabel="RMS error (clipped at 0.6)")
        best = int(np.argmin(errs))
        rows.append([lam, alphas[best], round(errs[best], 3)])
        rlhud.progress(i, len(lams), "λ values")
    rlhud.table(rows, headers=["λ", "best α", "RMS at best α"], title="Best step size per λ")

    # What a trace looks like: one episode, lam = 0.8, snapshot of e after every step
    env, lam = RandomWalk(), 0.8
    random.seed(3)
    e, s = np.zeros(env.n_states), env.reset()[0]
    snapshots = []
    while True:
        s2, r, term, trunc, _ = env.step(coin_flip(s))
        e[s] += 1.0
        snapshots.append(e[1:-1].copy())
        e *= lam
        if term or len(snapshots) >= 12:
            break
        s = s2
    rlhud.grid(np.array(snapshots), title="Accumulating trace e(s) per step (rows = time, cols = states 1–19)",
               cmap="heat", fmt=".1f")

In [ ]:
#@title ✅ **Check** · TD(λ) with eligibility traces (+120 XP) — tests the code you last ran above
rlcolab.check("07-nstep-and-planning/02-td-lambda")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

It is the TD(0) loop from lesson 6 with two extra lines: the trace bump before the update and the decay after. The update itself becomes a numpy vector operation over the whole `V`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Order matters. Compute δ first (using the *old* V), then `e[s] += 1`, then `V += alpha * delta * e`, then `e *= gamma * lam`. Swap the bump and the update and the current state gets no credit for its own TD error.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

while True:
    s2, r, terminated, truncated, _ = env.step(policy(s))
    delta = r + (0.0 if terminated else gamma * V[s2]) - V[s]
    e[s] += 1.0
    V += alpha * delta * e
    e *= gamma * lam
    if terminated or truncated: break
    s = s2

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("07-nstep-and-planning/02-td-lambda")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 07-nstep-and-planning/02-td-lambda
# Reference solution
import random

import numpy as np
import rlhud


def td_lambda(env, policy, n_episodes, lam, alpha, gamma=1.0, init=0.0, V=None):
    """Online TD(lambda) prediction with ACCUMULATING eligibility traces.

    Returns V (numpy array, length env.n_states); if V is given it is updated in place.
    Every step:
        delta = R + gamma * V(S') - V(S)        (V(S') counts as 0 if terminated)
        e(S) += 1                               (bump the trace of the state just left)
        V    += alpha * delta * e               (EVERY state, in proportion to its trace)
        e    *= gamma * lam                     (all traces fade)
    Traces start at zero at the beginning of every episode.
    """
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        e = np.zeros(env.n_states)
        s, _ = env.reset()
        while True:
            s2, r, terminated, truncated, _ = env.step(policy(s))
            delta = r + (0.0 if terminated else gamma * V[s2]) - V[s]
            e[s] += 1.0
            V += alpha * delta * e
            e *= gamma * lam
            if terminated or truncated:
                break
            s = s2
    return V


# ---------------------------------------------------------------- provided
class RandomWalk:
    """States 1..n are non-terminal, 0 and n+1 terminal; start in the middle.
    Action 0 = left, 1 = right. Exit right -> right_reward, exit left -> left_reward."""

    def __init__(self, n=19, left_reward=-1.0, right_reward=1.0):
        self.n = n
        self.n_states = n + 2
        self.n_actions = 2
        self.left_reward, self.right_reward = left_reward, right_reward

    def reset(self, seed=None):
        self.s = (self.n + 1) // 2
        return self.s, {}

    def step(self, a):
        self.s += 1 if a == 1 else -1
        if self.s == 0:
            return 0, self.left_reward, True, False, {}
        if self.s == self.n + 1:
            return self.s, self.right_reward, True, False, {}
        return self.s, 0.0, False, False, {}

    def true_values(self):
        k = np.arange(1, self.n + 1)
        return self.left_reward + (self.right_reward - self.left_reward) * k / (self.n + 1)


def coin_flip(s):
    return random.randrange(2)


def avg_rms(lam, alpha, runs=50, episodes=10, seed=0):
    """Mean RMS error over the first `episodes` episodes and `runs` runs on the 19-state walk."""
    env = RandomWalk()
    truth = env.true_values()
    random.seed(seed)
    total = 0.0
    for _ in range(runs):
        V = np.zeros(env.n_states)
        for _ in range(episodes):
            td_lambda(env, coin_flip, 1, lam, alpha, V=V)
            total += np.sqrt(np.mean((V[1:-1] - truth) ** 2))
    return total / (runs * episodes)


if __name__ == "__main__":
    alphas = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
    lams = [0.0, 0.4, 0.8, 0.9, 0.95, 1.0]
    rows = []
    for i, lam in enumerate(lams):
        with np.errstate(all="ignore"):                       # large α with λ ≈ 1 can diverge
            errs = [avg_rms(lam, a) for a in alphas]
        errs = [e if np.isfinite(e) and e < 0.6 else 0.6 for e in errs]   # clip for a readable plot
        rlhud.plot(errs, x=alphas, label=f"λ = {lam}", title="TD(λ) on the 19-state walk: avg RMS, first 10 episodes",
                   xlabel="α", ylabel="RMS error (clipped at 0.6)")
        best = int(np.argmin(errs))
        rows.append([lam, alphas[best], round(errs[best], 3)])
        rlhud.progress(i, len(lams), "λ values")
    rlhud.table(rows, headers=["λ", "best α", "RMS at best α"], title="Best step size per λ")

    # What a trace looks like: one episode, lam = 0.8, snapshot of e after every step
    env, lam = RandomWalk(), 0.8
    random.seed(3)
    e, s = np.zeros(env.n_states), env.reset()[0]
    snapshots = []
    while True:
        s2, r, term, trunc, _ = env.step(coin_flip(s))
        e[s] += 1.0
        snapshots.append(e[1:-1].copy())
        e *= lam
        if term or len(snapshots) >= 12:
            break
        s = s2
    rlhud.grid(np.array(snapshots), title="Accumulating trace e(s) per step (rows = time, cols = states 1–19)",
               cmap="heat", fmt=".1f")

> **⚠️ WARNING** · **Accumulating traces can blow up**
>
> A state revisited many times in quick succession builds a trace above 1, and with large α the
> updates overshoot — you will see λ = 1 diverge in the demo. *Replacing* traces (set e(s) = 1
> instead of adding 1) and the modern *true online TD(λ)* fix this. With function approximation
> (next chapter) the same idea uses a trace vector over weights.

## 03 · Models and planning

So far every update came straight from real experience. Now suppose the agent also learns a
**model**: a prediction of what the environment does.

| kind | gives you | example |
|---|---|---|
| **distribution model** | all outcomes and their probabilities, p(s', r ∣ s, a) | what DP needed in lesson 4 |
| **sample model** | one plausible outcome (s', r) when asked | a simulator; a table of what happened last time |

**Planning** means computing values or policies from a model instead of from the real world. DP
is planning with a distribution model. But planning does not need sweeps: you can **sample a
transition from the model and apply an ordinary Q-learning update to it**. To the learning rule,
a simulated transition looks exactly like a real one.

That unifies everything: *learning* and *planning* are the same backup applied to different
sources of experience.

## 04 · Dyna-Q

Dyna-Q (Sutton, 1990) runs both, interleaved, on one Q table. After every real step:

1. **act** ε-greedily in the real environment and observe R, S'
2. **direct RL** — Q-learning update from the real transition
3. **model learning** — remember `model[S, A] = (R, S')` (deterministic world: just a table)
4. **planning** — repeat n times: pick a previously seen (s, a) at random, ask the model for
   (r, s'), apply the Q-learning update

```
            ┌──────────── real experience ────────────┐
            ▼                                          │
   environment ──► direct RL update ──► Q ──► ε-greedy action
            │                           ▲
            ▼                           │
       model learning ──► model ──► planning updates (× n)
```

Why does this help so much? In a maze with a single reward at the goal, Q-learning's first
episode ends with **one** useful update: the step into the goal. In the second episode, the value
creeps back one more step, and so on. With planning, that single rewarding transition is
replayed, and the states leading to it — all remembered from the first episode — get updated in
**imagination**. Every real step of the second episode triggers n more replays, so the value
spreads back over most of the path within that single episode.

Here is plain Q-learning on a small maze — watch how long it takes for values to reach the start:

In [ ]:
#@title 🎮 **Interactive** · Live TD control
rlcolab.widget("qlearning", map=["......#G", "..#...#.", "S.#.....", "..#..#..", "........"], algo="q", alpha=0.5, gamma=0.95, epsilon=0.1)

---
### 🧪 Exercise 3 · Dyna-Q on a maze · ⭐ **170 XP**

Combine learning and planning: every real step also trains a model, and the model is replayed *n* times to squeeze more out of that experience.

1. `q_update(Q, s, a, r, s2, terminated, alpha, gamma)` — the Q-learning update, in place.
2. `plan(Q, model, n, rng, alpha, gamma)` — `model` is a dict `{(s, a): (r, s2, terminated)}`. Repeat *n* times: pick a remembered key uniformly at random (`keys = list(model)`, `keys[rng.integers(len(keys))]`), look up its outcome and apply `q_update`. Do nothing if the model is empty.
3. `dyna_q(env, n_episodes, n_planning, alpha=0.1, gamma=0.95, epsilon=0.1, seed=0)` → `(Q, steps, model)`. With `rng = np.random.default_rng(seed)`, a zero Q table and an empty model, every real step does: **(a)** act ε-greedily (provided `epsilon_greedy`), **(b)** `q_update` from the real transition, **(c)** `model[(s, a)] = (r, s2, terminated)`, **(d)** `plan(...)` with `n_planning` steps. `steps[i]` is the number of real steps in episode *i*.

The maze is Sutton & Barto's Figure 8.2 (`make_maze()`). The check compares `n_planning = 0` (plain Q-learning) with `n_planning = 10` over 5 seeds. **Run** plots steps per episode for n = 0, 5, 50 and shows how far the value of the goal has spread after only two episodes.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 07-nstep-and-planning/03-dyna-q
import numpy as np
import rlhud
from rlenvs import GridWorld


def q_update(Q, s, a, r, s2, terminated, alpha, gamma):
    """Q-learning update in place: Q[s,a] += alpha * (r + gamma * max Q[s2] - Q[s,a]) (no bootstrap if terminated)."""
    # TODO
    raise NotImplementedError


def plan(Q, model, n, rng, alpha, gamma):
    """n planning updates. Each one: pick a previously seen (s, a) uniformly at random from `model`,
    look up the remembered (r, s2, terminated) and apply q_update as if it had just happened."""
    # TODO
    raise NotImplementedError


def dyna_q(env, n_episodes, n_planning, alpha=0.1, gamma=0.95, epsilon=0.1, seed=0):
    """Tabular Dyna-Q (Sutton & Barto, section 8.2).

    Returns (Q, steps, model):
      Q     : (n_states, n_actions) array
      steps : list, number of real environment steps taken in each episode
      model : dict {(s, a): (r, s2, terminated)} — the last outcome seen for each pair
    """
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    model = {}
    steps = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        n = 0
        # TODO: each real step: (a) act epsilon-greedily, (b) q_update, (c) update the model,
        #       (d) plan with n_planning steps. Stop on terminated or truncated; record n in `steps`.
        raise NotImplementedError
    return Q, steps, model


# ---------------------------------------------------------------- provided
def epsilon_greedy(Q, s, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))


# The maze of Sutton & Barto Figure 8.2: 6 x 9, start on the left, goal top-right. Shortest path: 14 steps.
MAZE = [
    ".......#G",
    "..#....#.",
    "S.#....#.",
    "..#......",
    ".....#...",
    ".........",
]


def make_maze():
    return GridWorld(MAZE, step_reward=0.0, rewards={"G": 1.0}, max_steps=5000)


if __name__ == "__main__":
    EPISODES, RUNS = 50, 10
    for n_plan in (0, 5, 50):
        curve = np.zeros(EPISODES)
        for run in range(RUNS):
            env = make_maze()
            _, steps, _ = dyna_q(env, EPISODES, n_plan, seed=run)
            curve += np.array(steps) / RUNS
        rlhud.plot(curve[1:], x=list(range(2, EPISODES + 1)), label=f"{n_plan} planning steps",
                   title="Dyna maze: steps per episode (avg of 10 runs)", xlabel="episode", ylabel="steps")
        rlhud.stat(f"n={n_plan}: steps in episode 2", round(float(curve[1]), 1))

    # Policy after only 2 episodes: with planning, most of the maze already points to the goal
    for n_plan in (0, 50):
        env = make_maze()
        Q, _, model = dyna_q(env, 2, n_plan, seed=0)
        arrows = env.policy_grid(Q)
        for st in range(env.n_states):            # no arrow where nothing has been learned yet
            if Q[st].max() <= 0:
                r, c = env.pos_of(st)
                arrows[r][c] = None
        rlhud.grid(env.to_grid(Q.max(axis=1)), title=f"max Q after 2 episodes, n = {n_plan}",
                   policy=arrows,
                   walls=env.walls_mask(), labels=env.labels(), cmap="heat")
    print("Planning re-uses every real transition many times: the value of the goal spreads in a few episodes.")

In [ ]:
#@title ✅ **Check** · Dyna-Q on a maze (+170 XP) — tests the code you last ran above
rlcolab.check("07-nstep-and-planning/03-dyna-q")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

The maze only rewards reaching G (+1), so the first episode is a long random walk for everyone. Planning pays off from episode 2: the single rewarding transition gets replayed and its value propagates backwards through the remembered transitions.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

The model of a deterministic env is just a lookup table: overwrite `model[(s, a)]` every time you take `a` in `s`. Planning uses the same `q_update` as direct RL — the only difference is where the transition comes from.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Inside the episode loop:

a = epsilon_greedy(Q, s, epsilon, rng)
s2, r, terminated, truncated, _ = env.step(a); n += 1
q_update(Q, s, a, r, s2, terminated, alpha, gamma)
model[(s, a)] = (r, s2, terminated)
plan(Q, model, n_planning, rng, alpha, gamma)
if terminated or truncated: break
s = s2

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("07-nstep-and-planning/03-dyna-q")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 07-nstep-and-planning/03-dyna-q
# Reference solution
import numpy as np
import rlhud
from rlenvs import GridWorld


def q_update(Q, s, a, r, s2, terminated, alpha, gamma):
    """Q-learning update in place: Q[s,a] += alpha * (r + gamma * max Q[s2] - Q[s,a]) (no bootstrap if terminated)."""
    target = r if terminated else r + gamma * np.max(Q[s2])
    Q[s, a] += alpha * (target - Q[s, a])


def plan(Q, model, n, rng, alpha, gamma):
    """n planning updates. Each one: pick a previously seen (s, a) uniformly at random from `model`,
    look up the remembered (r, s2, terminated) and apply q_update as if it had just happened."""
    if not model:
        return
    keys = list(model)
    for _ in range(n):
        s, a = keys[rng.integers(len(keys))]
        r, s2, terminated = model[(s, a)]
        q_update(Q, s, a, r, s2, terminated, alpha, gamma)


def dyna_q(env, n_episodes, n_planning, alpha=0.1, gamma=0.95, epsilon=0.1, seed=0):
    """Tabular Dyna-Q (Sutton & Barto, section 8.2).

    Returns (Q, steps, model):
      Q     : (n_states, n_actions) array
      steps : list, number of real environment steps taken in each episode
      model : dict {(s, a): (r, s2, terminated)} — the last outcome seen for each pair
    """
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    model = {}
    steps = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        n = 0
        while True:
            a = epsilon_greedy(Q, s, epsilon, rng)                      # (a) act
            s2, r, terminated, truncated, _ = env.step(a)
            n += 1
            q_update(Q, s, a, r, s2, terminated, alpha, gamma)          # (b) direct RL
            model[(s, a)] = (r, s2, terminated)                         # (c) model learning
            plan(Q, model, n_planning, rng, alpha, gamma)               # (d) planning
            if terminated or truncated:
                break
            s = s2
        steps.append(n)
    return Q, steps, model


# ---------------------------------------------------------------- provided
def epsilon_greedy(Q, s, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))


# The maze of Sutton & Barto Figure 8.2: 6 x 9, start on the left, goal top-right. Shortest path: 14 steps.
MAZE = [
    ".......#G",
    "..#....#.",
    "S.#....#.",
    "..#......",
    ".....#...",
    ".........",
]


def make_maze():
    return GridWorld(MAZE, step_reward=0.0, rewards={"G": 1.0}, max_steps=5000)


if __name__ == "__main__":
    EPISODES, RUNS = 50, 10
    for n_plan in (0, 5, 50):
        curve = np.zeros(EPISODES)
        for run in range(RUNS):
            env = make_maze()
            _, steps, _ = dyna_q(env, EPISODES, n_plan, seed=run)
            curve += np.array(steps) / RUNS
        rlhud.plot(curve[1:], x=list(range(2, EPISODES + 1)), label=f"{n_plan} planning steps",
                   title="Dyna maze: steps per episode (avg of 10 runs)", xlabel="episode", ylabel="steps")
        rlhud.stat(f"n={n_plan}: steps in episode 2", round(float(curve[1]), 1))

    # Policy after only 2 episodes: with planning, most of the maze already points to the goal
    for n_plan in (0, 50):
        env = make_maze()
        Q, _, model = dyna_q(env, 2, n_plan, seed=0)
        arrows = env.policy_grid(Q)
        for st in range(env.n_states):            # no arrow where nothing has been learned yet
            if Q[st].max() <= 0:
                r, c = env.pos_of(st)
                arrows[r][c] = None
        rlhud.grid(env.to_grid(Q.max(axis=1)), title=f"max Q after 2 episodes, n = {n_plan}",
                   policy=arrows,
                   walls=env.walls_mask(), labels=env.labels(), cmap="heat")
    print("Planning re-uses every real transition many times: the value of the goal spreads in a few episodes.")

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("07-nstep-and-planning#2", {"q": "Dyna-Q with n = 50 planning steps reaches good performance in far fewer episodes than n = 0. What does it cost?", "options": ["Nothing — it is strictly better", "More computation per real step (51 updates instead of 1), and it relies on the model being right", "It needs the true transition probabilities"], "answer": 1, "explain": "Planning trades compute for data. That is a great deal when real experience is expensive (robots, users) — and dangerous if the model is wrong."})

### When the model is wrong

Real worlds change. If a shortcut opens up or a path closes, the model still remembers the old
world, and planning confidently reinforces stale beliefs. **Dyna-Q+** adds an exploration bonus
to planning rewards, r + κ√τ, where τ is how long ago (s, a) was last tried for real. Pairs
not tested for a while start to look attractive, so the agent goes back and checks.

### Prioritized sweeping

Uniformly random planning wastes most updates: early on, almost every remembered transition has
δ = 0 because nothing around it has value yet. **Prioritized sweeping** keeps a priority queue of
(s, a) pairs keyed by the size of their expected update |δ|. After a real step with a big δ, it
updates that pair, then pushes its **predecessors** (pairs the model says lead into s) whose
priority is above a threshold, and keeps popping the largest. Value changes ripple backwards
from where something actually changed — the same "work backwards from the goal" intuition as DP,
but focused. On mazes it often needs 5–10× fewer updates than Dyna-Q.

## 05 · Pitfalls

> **⚠️ WARNING** · **Common bugs in this lesson**
>
> - **Off-by-one in n-step indexing.** The rewards for τ are R_τ+1 … R_min(τ+n, T). Store a dummy
>   R₀ so list indices match the maths.
> - **Forgetting the tail.** After the episode ends, the last n − 1 states still need updates.
>   Keep looping until τ = T − 1.
> - **Bootstrapping past the end.** If τ + n ≥ T there is no V(S_τ+n) term.
> - **Traces not reset.** Eligibility traces belong to one episode; zero them at every reset.
> - **Planning on a separate table.** Dyna's planning updates must hit the *same* Q the agent acts on.
> - **argmax ties in a sparse-reward maze.** All-zero Q plus `np.argmax` means "always UP": break ties at random.

## 06 · Summary

- **n-step returns** use n real rewards then bootstrap; n is a bias–variance dial between TD and MC,
  and intermediate n usually learns fastest.
- **TD(λ)** averages all n-step returns; **eligibility traces** implement it online by sharing each
  TD error among recently visited states.
- A **model** lets the agent generate simulated experience; **planning** is just learning applied to it.
- **Dyna-Q** interleaves acting, learning, model learning and n planning updates per step — far
  fewer real episodes for the same performance. **Prioritized sweeping** spends planning where values change.

This closes the tabular chapter. Everything so far stored one number per state or state–action
pair. Next we replace the table with a function — and the ideas you now own (TD errors, targets,
bootstrapping, replay of experience) carry over almost unchanged.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("07-nstep-and-planning")